# Домашнее задание: RAG по созвездиям + Image-RAG

**Часть 1 (7 баллов):** текстовый RAG по статьям Wikipedia о созвездиях с Qwen.

**Часть 2 (3 балла):** мультимодальный RAG, который отвечает на вопросы по картинке через BLIP captioning + текстовый RAG.

**Общая задача:** построить RAG систему на основе статей Wiki, которая сможет отвечать на вопросы о созвездиях, а также о различных изображениях, где встречаются отсылки к созвездиям. В идеале система должна быть способна отвечать как на специфичные вопросы, касающиеся размера, количества звезд в созвездии, так и на простые (например, "Какие созвездия включают в себя изображения человека?")

Ссылка для сдачи ДЗ: https://forms.gle/NH7m25G6X9qYX7j38

## 0. Установка зависимостей

In [ ]:
!pip install -q 'numpy<2'
!pip install -q wikipedia-api
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q chromadb
!pip install -q sentence-transformers
!pip install -q transformers accelerate
!pip install -q pillow

import os
os.kill(os.getpid(), 9)

In [ ]:
import os
import re
import warnings  # чтобы заглушить лишние предупреждения
import logging  # для логирования MultiQueryRetriever
from typing import List, Dict, Tuple
from pprint import pprint

warnings.filterwarnings('ignore')
# Отключаем телеметрию ChromaDB
os.environ['ANONYMIZED_TELEMETRY'] = 'False'
# Отключаем параллелизм токенизаторов, чтобы не было предупреждений после fork()
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

---
# Часть 1. Текстовый RAG по созвездиям (7 баллов)

Мы построим RAG-систему, которая отвечает на вопросы о созвездиях, опираясь на статьи Wikipedia.

Pipeline стандартный: **парсинг → чанкинг → эмбеддинги → векторное хранилище → retrieval → генерация**.

## 1.1. Парсинг Wikipedia (1 балл)

Необходимо скачать английские статьи через `wikipedia-api` о созвездиях. API сам отдаёт чистый текст без вики-разметки.


In [ ]:
import wikipediaapi  # клиент к Wikipedia API

# Создаём клиент с user-agent, Wikipedia требует его указывать
wiki = wikipediaapi.Wikipedia(
    user_agent='RAG-Homework/1.0 (educational)',  # идентификатор
    language='en',  # английская Wikipedia
    extract_format=wikipediaapi.ExtractFormat.WIKI  # вернёт plain-text без HTML
)


CONSTELLATIONS = [
    "Orion (constellation)",
    "Scorpius",
    "Ursa Major",
    "Cassiopeia (constellation)",
    "Leo (constellation)",
    "Cygnus (constellation)",
    "Gemini (constellation)",
    "Aquarius (constellation)",
    "Perseus (constellation)",
    "Virgo (constellation)",
    "Andromeda (constellation)",
    "Pegasus (constellation)",
    "Hercules (constellation)",
    "Lyra",
    "Centaurus",
    "Crux",
    "Aries (constellation)",
    "Taurus (constellation)",
    "Cancer (constellation)",
    "Sagittarius (constellation)",
    "Capricornus",
    "Pisces (constellation)",
    "Libra (constellation)"
]

def fetch_wiki_article(page_title: str) -> str:
    """Скачивает текст статьи Wikipedia и аккуратно обрабатывает ошибки."""
    # Запрашиваем страницу
    page = wiki.page(page_title)
    # Проверяем существование страницы
    if not page.exists():
        # Прописываем исключение
        raise ValueError(f"Страница '{page_title}' не найдена в Wikipedia")
    text = page.text
    # Убираем повторяющиеся переводы строк
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text

#ВАШ КОД
# Качаем статьи по всем созвездиям с помощью функции fetch_wiki_articles, сохраняем в словарь articles


In [ ]:
# Смотрим первые 500 символов каждой статьи, чтобы удостовериться, что все верно
for name, text in articles.items():
    print(f'=== {name} ===')
    print(text[:500])
    print('...\n')

## 1.2. Сравнение двух стратегий чанкования (3 балла)

В RAG способ разбиения текста на куски (чанки) напрямую влияет на качество retrieval. Сравним два сплиттера из LangChain:

- **`CharacterTextSplitter`** - режет по одному разделителю (по умолчанию `\n\n`). Простой, но может выдать слишком длинные или слишком короткие куски, если разделитель распределён неравномерно.
- **`RecursiveCharacterTextSplitter`** - пробует разделители по списку (от `\n\n` к `\n` к ` ` к ``). Это даёт более ровный размер чанков и сохраняет смысловые границы.

Необходимо в этой части:
*   дописать функцию для подсчета простой статистики для сравнения чанкеров - 0,5 балла
*   проанализировать 2 вида чанкеров с разными параметрами и ответить на поставленные вопросы- 2 балла
*   реализовать отдельную функцию для сравнения чанкеров - 0,5 балла





In [ ]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
# Параметры одинаковые для обоих, чтобы сравнение было честным
CHUNK_SIZE = 800  # целевой размер чанка в символах
CHUNK_OVERLAP = 100  # перекрытие помогает не терять контекст на границах

char_splitter = CharacterTextSplitter(
    separator='\n\n',  # единственный разделитель
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,  # считаем длину в символах, а не в токенах
)

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=['\n\n', '\n', '. ', ' ', ''],  # от смысловых границ к посимвольным
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
)

# Берём для сравнения текст про Орион
sample_text = articles['Orion']

# Применяем оба сплиттера к одному и тому же тексту
char_chunks = char_splitter.split_text(sample_text)
recursive_chunks = recursive_splitter.split_text(sample_text)

# Напишите функцию для подсчета простой статистики: количество кусков, средняя/мин/макс длина
def chunk_stats(chunks: List[str]) -> Dict[str, float]:
    #ВАШ КОД

print('CharacterTextSplitter:', chunk_stats(char_chunks))
print('RecursiveCharacterTextSplitter:', chunk_stats(recursive_chunks))

In [ ]:
# Визуальное сравнение: рисуем гистограмму длин для обоих сплиттеров рядом
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Гистограмма длин для CharacterTextSplitter
axes[0].hist([len(c) for c in char_chunks], bins=20, color='steelblue', edgecolor='black')
axes[0].set_title(f'CharacterTextSplitter (n={len(char_chunks)})')
axes[0].set_xlabel('Длина чанка (символы)')
axes[0].set_ylabel('Количество')
# Вертикальная линия на целевом размере. Так мы увидим, насколько сплиттер от него отклоняется
axes[0].axvline(CHUNK_SIZE, color='red', linestyle='--', label=f'target={CHUNK_SIZE}')
axes[0].legend()

# Гистограмма для RecursiveCharacterTextSplitter
axes[1].hist([len(c) for c in recursive_chunks], bins=20, color='seagreen', edgecolor='black')
axes[1].set_title(f'RecursiveCharacterTextSplitter (n={len(recursive_chunks)})')
axes[1].set_xlabel('Длина чанка (символы)')
axes[1].set_ylabel('Количество')
axes[1].axvline(CHUNK_SIZE, color='red', linestyle='--', label=f'target={CHUNK_SIZE}')
axes[1].legend()

plt.tight_layout()
plt.show()

### TO DO

Поэкспериментируйте со сплиттерами. В ячейке ниже сравните, как меняется распределение длин чанков при разных значениях `chunk_size` (например, 400 vs 800 vs 1600) и `chunk_overlap` (0 vs 100 vs 300).

**Ответьте на следующие вопросы:**
1. Какой сплиттер вы бы выбрали для дальнейшей работы и почему?
2. При каких `chunk_size` и `chunk_overlap` распределение выглядит самым ровным?
3. Что произойдёт с retrieval, если поставить `chunk_size=100`? А `chunk_size=4000`?

In [ ]:
# ВАШ КОД

**Ваш ответ:**

_(напишите тут наблюдения)_

## 1.3. Чанкование всех статей с метаданными

Идём дальше с `RecursiveCharacterTextSplitter` (для всех статей). К каждому чанку привязываем метаданные:
- `constellation` — название созвездия, чтобы можно было фильтровать или цитировать источник;
- `chunk_id` — порядковый номер чанка внутри статьи;
- `source` — точное название Wikipedia-страницы.

In [ ]:
from langchain_core.documents import Document

# Собираем все чанки со всех статей в один список документов
all_documents: List[Document] = []

# Создаём пустой список, в который будем складывать все документы-чанки
all_documents = []

# Проходим по каждой статье: берём название созвездия и полный текст статьи
for constellation_name, text in articles.items():

    # Разбиваем полный текст статьи на небольшие фрагменты с помощью чанкера, который вы выбрали (тут recursive)
    chunks = recursive_splitter.split_text(text)
    # Проходим по всем чанкам текущей статьи
    # idx — номер чанка внутри статьи
    # chunk — текст конкретного чанка
    for idx, chunk in enumerate(chunks):

        # Создаём объект Document для одного чанка
        # page_content хранит сам текст чанка
        # metadata хранит дополнительную информацию о происхождении чанка
        doc = Document(
            page_content=chunk,
            metadata={
                # Название созвездия, к которому относится этот чанк
                'constellation': constellation_name,
                # Номер чанка внутри исходной статьи
                'chunk_id': idx,
                # Точное название страницы Wikipedia, откуда была взята статья
                'source': CONSTELLATIONS[constellation_name],
            }
        )

        # Добавляем созданный документ в общий список
        all_documents.append(doc)
print(f'Всего чанков: {len(all_documents)}')
# Проверяем, что всё работает
print('\nПример первого чанка:')
print('Metadata:', all_documents[0].metadata)
print('Content[:300]:', all_documents[0].page_content[:300])

## 1.4. Эмбеддинги и векторное хранилище ChromaDB


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Загружаем модель эмбеддингов
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True},
)

# Создаём векторное хранилище и индексируем все наши документы
# Под капотом каждый chunk прогоняется через embeddings и кладётся в Chroma
vectorstore = Chroma.from_documents(
    documents=all_documents,  # список документов из предыдущей ячейки
    embedding=embeddings,  # наша модель эмбеддингов
    collection_name='constellations',  # имя коллекции внутри Chroma
)

print(f'Проиндексировано: {vectorstore._collection.count()} чанков')

In [ ]:
# Ищем 3 ближайших чанка к простому запросу
test_query = 'What is the brightest star in Orion?'
results = vectorstore.similarity_search(test_query, k=3)
for i, doc in enumerate(results):
    # Смотрим, что нашлось что-то разумное
    print(f'--- Результат #{i+1} (constellation={doc.metadata["constellation"]}, chunk_id={doc.metadata["chunk_id"]}) ---')
    print(doc.page_content[:300])
    print()

--- Результат #1 (constellation=Orion, chunk_id=8) ---
Bright stars
Betelgeuse, also designated Alpha Orionis, is a massive M-type red supergiant star nearing the end of its life. It is the second-brightest star in Orion, and is a semiregular variable star. It serves as the right shoulder of the hunter (assuming that he is facing the observer). It is ge

--- Результат #2 (constellation=Orion, chunk_id=1) ---
Orion is most prominent during winter evenings in the Northern Hemisphere, as are five other constellations that have stars in the Winter Hexagon asterism. Orion's two brightest stars, Rigel (β) and Betelgeuse (α), are both among the brightest stars in the night sky; both are supergiants and slightl

--- Результат #3 (constellation=Orion, chunk_id=15) ---
Orion's Belt, or The Belt of Orion, is an asterism within the constellation. It consists of three bright stars: Alnitak (Zeta Orionis), Alnilam (Epsilon Orionis), and Mintaka (Delta Orionis). Alnitak is around 800 light-years away

## 1.5. LLM для генерации ответов: Qwen (1,5 балла)

Берём `Qwen/Qwen2.5-0.5B-Instruct`,  на CPU она работает медленно, но работает. Если у вас работает медленно, то используйте с явным `torch_dtype=torch.float32` и небольшим `max_new_tokens`.

Альтернативно можете подключить любую другую модель через HF pipeline.

Что необходимо сделать:

1.  Соберите pipeline для генерации текста с моделью Qwen, которую мы загрузили.[Тут можете посмотреть информацию о pipeline](https://huggingface.co/docs/transformers/main_classes/pipelines#transformers.TextGenerationPipeline)

выше (model, tokenizer) - 1 балл

Требования к параметрам:

*   Task - 'text-generation'
*   Уже загруженные model и tokenizer
*   Максимум новых токенов. Определите ограничение длины ответа модели. Чем меньше, тем быстрее на CPU, но можно обрезать ответ на середине.
*   Отключите сэмплирование (детерминированная генерация), чтобы при повторных запусках получать одинаковый ответ
*   Штраф за повторы: значения >1 наказывают модель за повторение уже сгенерированных токенов, что помогает бороться с зацикливаниями маленьких моделей
*   Только ответ модели в выводе: настройте так, чтобы pipeline возвращал только сгенерированный текст без эхо промпта

2. Пропишите промпты для system и user промпты - 0,5 балла



In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

LLM_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float32,  # явно float32 для CPU
    device_map='cpu',  # явно указываем CPU
)

# Оборачиваем в pipeline для удобства

llm_pipeline = pipeline(
    'text-generation',
    #ВАШ КОД


In [ ]:
def generate_answer(question: str, context: str) -> str:
    """Собирает промпт из вопроса и контекста, прогоняет через Qwen и возвращает ответ."""
    # ДЛя модели нужен формат chat: [{'role': 'system', ...}, {'role': 'user', ...}]
    messages = [
        {
            'role': 'system',
            # System prompt задаёт роль и явно ограничивает модель только контекстом
            'content': (
                #ВАШ ПРОМПТ
            )
        },
        {
            'role': 'user',
            # Передаём контекст и вопрос в user-message — простой шаблон
            'content': #ВАШ ПРОМПТ
        }
    ]
    # Применяем chat-template токенизатора — он сам подставит спецтокены Qwen
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,  #pipeline сам токенизирует
        add_generation_prompt=True,  # добавит токен начала ответа ассистента
    )
    # Прогоняем через pipeline
    output = llm_pipeline(prompt)
    # Берём текст ответа и убираем хвостовые пробелы
    return output[0]['generated_text'].strip()

# Быстрая проверка
print(generate_answer(
    question='What is Orion?',
    context='Orion is a prominent constellation located on the celestial equator and visible throughout the world.'
))

## 1.6. Полный pipeline (0,5 баллов)

Объединяем retrieval и генерацию в одну функцию. Возвращаем не только ответ, но и **источники**. Это важно для отладки и для того, чтобы видеть, опирался ли LLM на правильный контекст.

Допишите функцию rag_query, которая собирает весь RAG пайплайн

In [ ]:
def rag_query(question: str, k: int = 4, verbose: bool = True) -> Dict:
    """Полный RAG-пайплайн: retrieval из Chroma → склейка контекста → генерация ответа."""
    # Шаг 1: получаем top-k наиболее похожих чанков по cosine similarity
    retrieved_docs = #ВАШ КОД
    # Шаг 2: склеиваем все найденные чанки в единый контекст
    # Двойной перевод строки между чанками, чтобы модель видела границы
    context = #ВАШ КОД
    # Шаг 3: вызываем LLM с собранным контекстом
    answer = #ВАШ КОД
    # Шаг 4: собираем структурированный результат для удобной отладки
    result = { #ВАШ КОД
    }
    # Функция распечатает вопрос, ответ и источники прямо в вывод ячейки, если verbose == True
    if verbose:
        #ВАШ КОД
    return result

## 1.7. Тестирование (1 балл)


1.   Придумайте 7 вопросов с разной сложностью и охватом, чтобы проверить, как справляется с ними RAG - 0,5 балла

Примеры:


*   Which constellations are part of the zodiac? -  требует знаний по всему корпусу
*   Compare Ursa Major and Scorpius in terms of visibility from the northern hemisphere. - сравнительный вопрос
*   How was Cassiopeia named in Greek mythology? - культурно-исторический контекст


2.   Проведите небольшие эксперименты и ответьте на вопросы - 0,5 балла


*   Что произойдёт, если поменять k=4 на k=1 и на k=10? Когда больше контекста, становится хуже?
*   На каких типах запросов RAG работает лучше: на локальных (один факт) или на агрегирующих (нужна информация из нескольких статей)? Почему?
*   Как справляется RAG с вопросами про созвездия, которых нет в базе данных?









In [ ]:
story_queries = [
    # ВАШИ ВОПРОСЫ
]


results_manual = []
for q in story_queries:
    result = rag_query(q, k=4, verbose=True)
    results_manual.append(result)

**Ваш ответ:**

_(напишите тут наблюдения)_

---
# Часть 2. Image-RAG: вопросы по картинке (3 балла)

Расширим систему до мультимодального RAG. Вам даны изображения созвездий в различных стилях, необходимо сделать так, чтобы система отвечала на вопросы о них.

Pipeline:
1. На входе изображение созвездия.
2. BLIP делает **caption** (текстовое описание).
3. Caption становится запросом к нашему RAG.
4. LLM отвечает на пользовательский вопрос, используя retrieval по caption.

Используем BLIP в двух режимах: **unconditional** (генерация описания с нуля) и **conditional** (генерация с подсказкой-префиксом).

Необходимо сделать:
* Дописать функцию generate_caption() - 0,5 балла
* Загрузить изображения и сгенерировать captions для 5-7 изображений из датасета - 0,5 балла
* Придумайте 5 вопросов к изображениям, чтобы протестировать Image-RAG - 0,5 балла
* Перепишите pipeline обычного RAG, чтобы он работал с изображениями. Пропишите новые промпты - 1 балл
* Ответьте на вопросы - 0,5 балла




In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from io import BytesIO

BLIP_MODEL_NAME = 'Salesforce/blip-image-captioning-base'

blip_processor = BlipProcessor.from_pretrained(BLIP_MODEL_NAME)
blip_model = BlipForConditionalGeneration.from_pretrained(
    BLIP_MODEL_NAME,
    torch_dtype=torch.float32,  # float32 для CPU
)
blip_model.eval()  # переводим в eval-режим

Что должна делать функция:

1.   Препроцессинг. Подготовьте входы для модели через blip_processor. BLIP
поддерживает два режима:


*   Unconditional — если conditional_prompt не задан: процессору передаётся только картинка
*   Conditional — если conditional_prompt задан: процессору передаётся И картинка, И текст-подсказка (BLIP будет продолжать caption с этой подсказки)

Пример и информацию про модель можно посмотреть[здесь](https://huggingface.co/Salesforce/blip-image-captioning-base)

В обоих случаях blip_processor(...) ожидает аргумент return_tensors='pt', чтобы вернуть PyTorch-тензоры.

2. Генерация. Вызовите blip_model.generate(...), передав туда подготовленные входы (распакуйте через `**inputs`).

Параметры генерации:

*   max_length=50 - caption обычно короткий, длиннее тратить время на CPU не нужно
*  num_beams=4 - beam search даёт результат чуть качественнее, чем greedy decoding

Важно: оберните вызов generate в `with torch.no_grad()`: мы не обучаем модель, градиенты считать не нужно.


3.   Декодирование. generate возвращает тензор с токенами, а нам нужна строка. Используйте `blip_processor.decode(...)`, передав ему первый элемент результата (out[0]) и аргумент `skip_special_tokens=True` (иначе в строке останутся служебные токены типа `<s>, </s>, [CLS]`).









In [ ]:
def generate_caption(image: Image.Image, conditional_prompt: str = None) -> str:
    """Генерирует описание картинки"""

    # Conditional mode - BLIP получает на вход начало предложения и продолжает его
    if conditional_prompt:
        inputs = #ВАШ КОД
    else:
        # Unconditional mode - модель генерирует описание полностью с нуля
        inputs = #ВАШ КОД
    # Генерируем без градиентов — экономим память и время
    #ВАШ КОД
    # Декодируем токены обратно в текст, skip_special_tokens убирает <s>, </s> и т.п.
    caption = #ВАШ КОД
    return caption

In [ ]:
 #Загрузите изображения из датасета и выберете 5-7 изображений
#ВАШ КОД

In [ ]:
# Сравнение unconditional и conditional captioning для одного изображения
#Протестируйте генерацию, а затем допишите функцию, чтобы применить к 5-7 изображениям
# Unconditional
caption_uncond = caption_image(test_image)
print(f'[Unconditional]: {caption_uncond}')

# Conditional: подсказываем тематический префикс, который помогает направить модель
# Префикс становится началом caption, модель его продолжает. Caption можно задать такой, чтобы отражал наш домен 'the constellation'
caption_cond_1 = caption_image(test_image, conditional_prompt=#ВАШ КОД)
print(f'[Conditional]: {caption_cond_1}')
# Второй conditional-префикс
caption_cond_2 = caption_image(test_image, conditional_prompt=#ВАШ КОД)
print(f'[Conditional2]: {caption_cond_2}')

### Вопросы: анализ качества caption

Запустите captioning на выбранных изображениях и сравните:
1. Оцените качество генерации caption с помощью BLIP для выбранных изображений
2. Помогает ли conditional prompt? В каких случаях?
3. Сравните два conditional prompts для нескольких изображений и оцените, какой сработал лучше всего в каких случаях.



## 2.2. Полный pipeline `image_rag_query`

Соединяем всё: картинка → BLIP caption → caption как запрос к Chroma → найденный контекст + caption + вопрос пользователя → ответ LLM.



In [ ]:
def image_rag_query(
    image: Image.Image,
    user_question: str,
    conditional_prompt: str = 'the constellation',
    k: int = 4,
    verbose: bool = True,
) -> Dict:
    """Мультимодальный RAG: картинка → caption → retrieval → ответ LLM."""
    # Шаг 1: BLIP делает caption картинки. Используем conditional — это даёт более релевантные описания
    caption = caption_image(image, conditional_prompt=conditional_prompt)
    if verbose:
        print(f'[BLIP caption]: {caption}')
    # Шаг 2: caption становится запросом к Chroma. Это ключевой мост между модальностями
    retrieved_docs = #ВАШ КОД
    # Шаг 3: склеиваем контекст
    context = #ВАШ КОД
    # Шаг 4: формируем кастомный промпт, в который кладём И caption, И вопрос пользователя
    # Это позволяет модели понимать, что вопрос относится к объекту на картинке
    messages = [
        {
            'role': 'system',
            'content': (
                #ВАШ КОД
            )
        },
        {
            'role': 'user',
            'content': (
                #ВАШ КОД
            )
        }
    ]
    # Применяем chat-template и генерируем ответ, как в generate_answer
    prompt = #ВАШ КОД
    output = #ВАШ КОД
    answer = #ВАШ КОД
    # Собираем результат
    result = {
        'caption': caption,  # что увидел BLIP
        'question': user_question,  # что спросил пользователь
        'answer': answer,  # что ответил LLM
        'sources': [  # источники из retrieval
            {
                'constellation': doc.metadata['constellation'],
                'chunk_id': doc.metadata['chunk_id'],
                'preview': doc.page_content[:150] + '...',
            }
            for doc in retrieved_docs
        ]
    }
    if verbose:
        #ВАШ КОД
    return result

### Вопросы: анализ качества Image RAG

1. Всегда ли caption приводит Image RAG к нужному созвездию?
2. Что произойдёт, если BLIP неправильно классифицирует картинку?

## Сhallenge: MultiQueryRetriever

Ручные формулировки вопросов работают, но иногда retrieval промахивается, потому что в чанках формулировка другая. `MultiQueryRetriever` решает эту проблему так: он просит LLM **перефразировать вопрос несколькими способами**, делает retrieval для каждой версии и объединяет результаты. В данном challenge вам необходимо сравнить ручной подход к перефразированию запросов для обычного RAG (не Image RAG) и встроенный в langchain.

**Пример:**

*Which constellation contains a famous red supergiant nearing the end of its life?*

В чанках нет "red supergiant" напрямую в контексте Ориона, нужно связать тип звезды с названием
    

Что необходимо сделать:

1.   Функция генерации перефразировок

Напишите функцию `generate_query_variants(question: str, n: int = 3) -> List[str]`, которая:

*   Формирует сообщения в chat-формате (system + user) для Qwen
*   В system prompt объясняет модели задачу: сгенерировать n разных формулировок одного и того же вопроса
*   В user сообщении передаёт исходный вопрос и просит вернуть ровно n вариантов, каждый на отдельной строке, без нумерации и пояснений
*   Прогоняет через llm_pipeline и парсит ответ: разбивает по \n, фильтрует пустые строки, возвращает список строк

2.   Функция `multiquery retrieval`

Напишите функцию multiquery_retrieve(question: str, n_variants: int = 3, k: int = 3) -> List[Document], которая:


*   Вызывает generate_query_variants чтобы получить n_variants перефразировок
*   Добавляет к ним исходный вопрос (итого n_variants + 1 запросов)
*   Для каждого запроса делает vectorstore.similarity_search(query, k=k)
*   Объединяет все найденные документы в один список
*   Убирает дубликаты. Oдин и тот же чанк мог найтись по нескольким вариантам запроса, поэтому стоит проверять
*   Возвращает список уникальных документов

3.   Полный pipeline
Напишите функцию `rag_query_multiquery(...)`

4.   Подключите **MultiQueryRetriever** из LangChain

Документация: https://python.langchain.com/docs/how_to/MultiQueryRetriever/

Вам понадобится:
*   импортировать MultiQueryRetriever из langchain.retrievers
*   обернуть llm_pipeline в HuggingFacePipeline из langchain_huggingface, так как LangChain не работает напрямую с HF pipeline
*   создать `MultiQueryRetriever.from_llm(retriever=..., llm=...)`
*   Включите логирование, чтобы видеть сгенерированные перефразировки

5. Ответьте кратко на вопросы

*   Одинаковые ли варианты с перефразированием генерирует Qwen в вашей реализации и в LangChain? Если разные, то предположите, по какой причине.
*   Как LangChain делает дедупликацию? Это отличается от вашего подхода?
*   Какой вариант даёт больше уникальных чанков и почему?
*   В каком сценарии вы бы выбрали свою реализацию вместо готовой из LangChain?





















